In [28]:
import requests
import pandas as pd
import time

BASE_URL = "https://api.deezer.com"

ARTISTS = [
    "Rulo y la contrabanda",
    "Heroes del Silencio",
    "El duende callejero",
    "Love of Lesbian",
    "Arde Bogota",
    "Mago de Oz",
    "Mr Kilombo",
    "Rozalen",
    "Taburete",
    "Extremoduro",
    "La Plazuela",
    "Veintiuno",
    "Ojete Calor",
    "Rata Blanca",
    "Vetusta Morla",
    "Leiva",
    "Bad Bunny",
    "Bunbury",
    "Travis Birds",
    "Joaquin Sabina",
    "Rosalia",
    "Queen",
    "The Lumineers",
    "Foo Fighters",
    "Muse",
    "Metallica",
    "Ginebras",
    "IZAL",
    "Kaiser Chiefs",
    "Residente"
]

def buscar_artista (name): 
    url = f"{BASE_URL}/search/artist" #define url con una base común para todas incluida en variable BASE_URL y un añadido para esta función
    params = {"q": name, "limit": 1} #diccionario para definir la busqueda, el nombre lo buscaremos desde su clave q(definida por deezer) limite 1 para que solo exporte 1 artista
    r = requests.get(url, params=params) # llamar a la url con los parámetros facilitados
    data = r.json()
    return data["data"][0] 

def buscar_albumes (id): #lo usamos para encontrar los album
    url = f"{BASE_URL}/artist/{id}/albums" #con la url base y el id obtenido en la función buscar_artista
    r = requests.get(url)
    data = r.json()
    return data["data"]

def buscar_nbtracks(id): #lo usamos para buscar temas dentro de los album, funciona con el id de album no de artista
    url = f"{BASE_URL}/album/{id}"
    r = requests.get(url)
    data = r.json()
    return data["nb_tracks"]

In [29]:
buscar_artista ("metallica")

{'id': 119,
 'name': 'Metallica',
 'link': 'https://www.deezer.com/artist/119',
 'picture': 'https://api.deezer.com/artist/119/image',
 'picture_small': 'https://cdn-images.dzcdn.net/images/artist/056578a9c2007f69ce198c81875eca41/56x56-000000-80-0-0.jpg',
 'picture_medium': 'https://cdn-images.dzcdn.net/images/artist/056578a9c2007f69ce198c81875eca41/250x250-000000-80-0-0.jpg',
 'picture_big': 'https://cdn-images.dzcdn.net/images/artist/056578a9c2007f69ce198c81875eca41/500x500-000000-80-0-0.jpg',
 'picture_xl': 'https://cdn-images.dzcdn.net/images/artist/056578a9c2007f69ce198c81875eca41/1000x1000-000000-80-0-0.jpg',
 'nb_album': 67,
 'nb_fan': 8052137,
 'radio': True,
 'tracklist': 'https://api.deezer.com/artist/119/top?limit=50',
 'type': 'artist'}

In [30]:
buscar_albumes (119)

[{'id': 428391407,
  'title': '72 Seasons',
  'link': 'https://www.deezer.com/album/428391407',
  'cover': 'https://api.deezer.com/album/428391407/image',
  'cover_small': 'https://cdn-images.dzcdn.net/images/cover/39aadc0fb5b3ca28614a2e064f7f7440/56x56-000000-80-0-0.jpg',
  'cover_medium': 'https://cdn-images.dzcdn.net/images/cover/39aadc0fb5b3ca28614a2e064f7f7440/250x250-000000-80-0-0.jpg',
  'cover_big': 'https://cdn-images.dzcdn.net/images/cover/39aadc0fb5b3ca28614a2e064f7f7440/500x500-000000-80-0-0.jpg',
  'cover_xl': 'https://cdn-images.dzcdn.net/images/cover/39aadc0fb5b3ca28614a2e064f7f7440/1000x1000-000000-80-0-0.jpg',
  'md5_image': '39aadc0fb5b3ca28614a2e064f7f7440',
  'genre_id': 152,
  'fans': 43754,
  'release_date': '2023-04-14',
  'record_type': 'album',
  'tracklist': 'https://api.deezer.com/album/428391407/tracks',
  'explicit_lyrics': False,
  'type': 'album'},
 {'id': 256261892,
  'title': 'The Metallica Blacklist',
  'link': 'https://www.deezer.com/album/256261892',

In [31]:
buscar_nbtracks (428391407)

12

In [32]:
results = []

for artista in ARTISTS: #si artista está en la lista que creamos al principio
    info_artista = buscar_artista (artista) #buscamos al artista llamando a la función y creamos variable
    print(f"Procesando: {artista}...") #ameniza un poco la espera enseñándonos a quien está procesando
    artist_id = info_artista ["id"] #extraemos su id con la variable anterior e indicando la clave que buscamos al ser diccionario
    albumes = buscar_albumes (artist_id) #llamamos a la funcion obtener_albumes con el id de artista obtenido en la anterior y creamos variable

    total = 0
    for album in albumes:
        album_id = album["id"] #buscamos el id llamando a la clave
        tracks = buscar_nbtracks (album_id) #buscamos el numero de canciones llamando a buscar_nbtracks con la id de album obtenida en el anterior
        total = total + tracks #sumamos a total el numero de canciones obtenidas
        time.sleep(0.1) #adicional, como deezer tiene un max de llamadas por seg, hacemos que python haga una pausa para no sobrecargar la API

    results.append({ #creamos un diccionario con los resultados obtenidos por artista
        "artista": artista,
        "total_canciones": total
    })
    print(f"✅ {artista}: {total} canciones") #nos indica que artista ha procesado ya y el total de canciones

df = pd.DataFrame(results) #convertir a tabla con pandas, va fuera del bucle porque si lo ponemos dentro imprime toda la lista con cada artista
df["supera_50"] = df["total_canciones"].apply(lambda x: "✅ Sí" if x >= 50 else "❌ No") 
#df supera 50 añade columna con ese nombre, apply asigna funcion a cada parte, lambda es una función rápida para el if/else
print(df)

Procesando: Rulo y la contrabanda...
✅ Rulo y la contrabanda: 146 canciones
Procesando: Heroes del Silencio...


KeyboardInterrupt: 